In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
import json
import dask
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from xcube.core.store import new_data_store
from xcube.core.chunk import chunk_dataset
from xcube.core.gridmapping import GridMapping
from xcube.core.geom import mask_dataset_by_geometry
from xcube_resampling.spatial import resample_in_space
from xcube_resampling.gridmapping import GridMapping
from dask.distributed import Client, LocalCluster

In [2]:
INPUT_DIR = "input_irrigation"

In [3]:
irr_store = new_data_store("file", root=INPUT_DIR)

In [4]:
bbox = [-5, 40, 3, 44] # Ebro Basin
time_range = ("2020-01-01", "2021-12-31")
# time_range = ("2020-01-01", "2020-01-31")

In [5]:
json_file_path = "credentials.json"
with open(json_file_path, "r") as j:
    credentials = json.loads(j.read())

In [7]:
import logging
logging.getLogger().setLevel(logging.WARNING)
LOG = logging.getLogger("xcube.clms")
LOG.handlers.clear() 
LOG.setLevel(logging.DEBUG)

handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)
handler.setFormatter(logging.Formatter(
    "%(asctime)s [%(levelname)s] %(name)s: %(message)s"
))
LOG.addHandler(handler)

In [8]:
%%time
clms_data_store = new_data_store("clms", credentials=credentials)

2025-09-02 14:55:51,912 [INFO] xcube.clms: Fetching datasets metadata from https://land.copernicus.eu/api
2025-09-02 14:55:51,913 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/api/@search/?portal_type=DataSet&fullobjects=1
2025-09-02 14:56:37,958 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/api/@search/?b_start=25&portal_type=DataSet&fullobjects=1
2025-09-02 14:57:26,064 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/api/@search/?b_start=50&portal_type=DataSet&fullobjects=1
2025-09-02 14:57:29,430 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/api/@search/?b_start=75&portal_type=DataSet&fullobjects=1
2025-09-02 14:57:32,525 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/api/@search/?b_start=100&portal_type=DataSet&fullobjects=1
2025-09-02 14:57:35,649 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/api/@search/?b_start=125&portal_type=DataSet&fullobjects=1
2025-09-0

CPU times: user 362 ms, sys: 117 ms, total: 479 ms
Wall time: 2min 6s


In [9]:
%%time
clms_data = clms_data_store.open_data("daily-surface-soil-moisture-v1.0", time_range=time_range)
clms_data

2025-09-02 14:58:05,195 [DEBUG] xcube.clms: Token expired or not present. Refreshing token.
2025-09-02 14:58:05,228 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/@@oauth2-token
2025-09-02 14:58:07,518 [DEBUG] xcube.clms: Token refreshed successfully.
2025-09-02 14:58:07,520 [DEBUG] xcube.clms: Token expired or not present. Refreshing token.
2025-09-02 14:58:07,551 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/@@oauth2-token
2025-09-02 14:58:08,443 [DEBUG] xcube.clms: Token refreshed successfully.
2025-09-02 14:58:08,444 [DEBUG] xcube.clms: Current token valid. Reusing it.
2025-09-02 14:58:08,444 [DEBUG] xcube.clms: Making a request to https://land.copernicus.eu/api/@get-download-file-urls/?dataset_uid=c073cf8a1d594593bc5d2f9024f0dc60&download_information_id=0524bf44-6624-4e94-b88a-38af27388b31&date_from=2014-01-01&date_to=2025-09-02
2025-09-02 14:58:12,691 [DEBUG] xcube.clms: Processing 31 files in batches of 20
2025-09-02 14:58:12,692 [DEBUG] 

CPU times: user 2.34 s, sys: 497 ms, total: 2.83 s
Wall time: 1min 5s


<xarray.Dataset> Size: 7GB
Dimensions:    (time: 31, lat: 4144, lon: 6832)
Coordinates:
  * lat        (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon        (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
  * time       (time) datetime64[ns] 248B 2020-01-01 2020-01-02 ... 2020-01-31
Data variables:
    crs        (time) |S1 31B b'' b'' b'' b'' b'' b'' ... b'' b'' b'' b'' b''
    ssm        (time, lat, lon) float32 4GB dask.array<chunksize=(1, 1382, 2278), meta=np.ndarray>
    ssm_noise  (time, lat, lon) float32 4GB dask.array<chunksize=(1, 1382, 2278), meta=np.ndarray>
Attributes: (12/26)
    Conventions:               CF-1.6
    archive_facility:          VITO
    copyright:                 Copernicus Service information 2020
    geospatial_lat_max:        72.0
    geospatial_lat_min:        35.0
    geospatial_lon_max:        50.0
    ...                        ...
    region_name:               CEURO
    sensor:                    CSAR
    source:                    Derived from EO radar observations
    time_coverage_end:         2020-01-01T23:59:59Z
    time_coverage_start:       2020-01-01T00:00:00Z
    title:                     Daily Surface Soil Moisture 1km: CEURO 2020-01...

In [ ]:
# Missing dates from source - '2020-07-18', '2021-05-31' when running this for 2 years from 2020-2021

In [11]:
clms_ssm_only = clms_data.drop_vars("ssm_noise") 
clms_ssm_only

<xarray.Dataset> Size: 4GB
Dimensions:  (time: 31, lat: 4144, lon: 6832)
Coordinates:
  * lat      (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon      (lon) float64 55kB -11.0 -10.99 -10.98 -10.97 ... 49.98 49.99 50.0
  * time     (time) datetime64[ns] 248B 2020-01-01 2020-01-02 ... 2020-01-31
Data variables:
    crs      (time) |S1 31B b'' b'' b'' b'' b'' b'' ... b'' b'' b'' b'' b'' b''
    ssm      (time, lat, lon) float32 4GB dask.array<chunksize=(1, 1382, 2278), meta=np.ndarray>
Attributes: (12/26)
    Conventions:               CF-1.6
    archive_facility:          VITO
    copyright:                 Copernicus Service information 2020
    geospatial_lat_max:        72.0
    geospatial_lat_min:        35.0
    geospatial_lon_max:        50.0
    ...                        ...
    region_name:               CEURO
    sensor:                    CSAR
    source:                    Derived from EO radar observations
    time_coverage_end:         2020-01-01T23:59:59Z
    time_coverage_start:       2020-01-01T00:00:00Z
    title:                     Daily Surface Soil Moisture 1km: CEURO 2020-01...

In [16]:
dask.config.set(scheduler="threads", num_workers=2)

In [17]:
%%time
irr_store.write_data(clms_ssm_only, f"clms.zarr")

CPU times: user 10.8 s, sys: 4.7 s, total: 15.5 s
Wall time: 24.3 s


'clms.zarr'

In [18]:
irr_store.list_data_ids()

['clms.zarr', 'era5.zarr', 'landcover2020global.zarr']

In [19]:
irr_store.open_data("clms.zarr")

<xarray.Dataset> Size: 7GB
Dimensions:  (time: 31, lat: 4144, lon: 6832)
Coordinates:
  * lat      (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon      (lon) float64 55kB -11.0 -10.99 -10.98 -10.97 ... 49.98 49.99 50.0
  * time     (time) datetime64[ns] 248B 2020-01-01 2020-01-02 ... 2020-01-31
Data variables:
    crs      (time) |S1 31B dask.array<chunksize=(31,), meta=np.ndarray>
    ssm      (time, lat, lon) float64 7GB dask.array<chunksize=(1, 1382, 2278), meta=np.ndarray>
Attributes: (12/26)
    Conventions:               CF-1.6
    archive_facility:          VITO
    copyright:                 Copernicus Service information 2020
    geospatial_lat_max:        72.0
    geospatial_lat_min:        35.0
    geospatial_lon_max:        50.0
    ...                        ...
    region_name:               CEURO
    sensor:                    CSAR
    source:                    Derived from EO radar observations
    time_coverage_end:         2020-01-01T23:59:59Z
    time_coverage_start:       2020-01-01T00:00:00Z
    title:                     Daily Surface Soil Moisture 1km: CEURO 2020-01...